In [62]:
#Importing Required Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [63]:
#Loading the Dataset
df = pd.read_csv("diabetes.csv")

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [64]:
#Exploring the dataset
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTarget Distribution:")
print(df["Outcome"].value_counts())

Dataset Shape: (768, 9)

Columns:
Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

Missing Values:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Target Distribution:
Outcome
0    500
1    268
Name: count, dtype: int64


In [65]:
#Splitting Features And Target
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

In [66]:
#Splitting Into Testing And Training
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [67]:
#Feature Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [68]:
#Training The KNN model Before Tuning
knn = KNeighborsClassifier()

knn.fit(X_train, y_train)

KNeighborsClassifier()

In [69]:
#Predicting Data And its Accuracy
y_pred = knn.predict(X_test)

default_accuracy = accuracy_score(y_test, y_pred)

print("Default Model Accuracy:", default_accuracy)

Default Model Accuracy: 0.6948051948051948


In [70]:
#Applying GribBasedCV Tuning
param_grid = {
    "n_neighbors": list(range(1, 21))
}

grid_search = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy"
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,
                                         13, 14, 15, 16, 17, 18, 19, 20]},
             scoring='accuracy')

In [71]:
# Display Best Hyperparameter and Best Cross Validation Score
print("Best Hyperparameter:", grid_search.best_params_)
print("Best Cross Validation Score:", grid_search.best_score_)

Best Hyperparameter: {'n_neighbors': 11}
Best Cross Validation Score: 0.7606024256963881


In [72]:
# Training The Model Using Best Hyperparameter
best_knn = grid_search.best_estimator_

best_knn.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=11)

In [73]:
#Predict Test Data and Calculate New Accuracy
grid_pred = best_knn.predict(X_test)

grid_accuracy = accuracy_score(y_test, grid_pred)

print("GridSearchCV Accuracy:", grid_accuracy)

GridSearchCV Accuracy: 0.7077922077922078


In [57]:
# Apply RandomizedSearchCV
param_dist = {
    "n_neighbors": list(range(1, 21))
}

random_search = RandomizedSearchCV(
    estimator=KNeighborsClassifier(),
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    random_state=42
)

random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5, estimator=KNeighborsClassifier(),
                   param_distributions={'n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8,
                                                        9, 10, 11, 12, 13, 14,
                                                        15, 16, 17, 18, 19,
                                                        20]},
                   random_state=42, scoring='accuracy')

In [58]:
# Display Best Hyperparameter and Best Cross Validation Score
print("Best Hyperparameter:", random_search.best_params_)
print("Best Cross Validation Score:", random_search.best_score_)

Best Hyperparameter: {'n_neighbors': 6}
Best Cross Validation Score: 0.757350393176063


In [59]:
# Train RandomizedSearchCV Best Model and Compare
random_knn = random_search.best_estimator_

random_knn.fit(X_train, y_train)

random_pred = random_knn.predict(X_test)

random_accuracy = accuracy_score(y_test, random_pred)

print("RandomizedSearchCV Accuracy:", random_accuracy)

RandomizedSearchCV Accuracy: 0.7012987012987013


In [60]:
# Comparison Table
comparison = pd.DataFrame({
    "Model": [
        "Default KNN",
        "GridSearchCV",
        "RandomizedSearchCV"
    ],
    "Accuracy": [
        default_accuracy,
        grid_accuracy,
        random_accuracy
    ]
})

comparison

,Model,Accuracy
0,Default KNN,0.694805
1,GridSearchCV,0.707792
2,RandomizedSearchCV,0.701299


In [61]:
# Best Model
best_model = comparison.loc[comparison["Accuracy"].idxmax()]

print("Best Performing Model:")
print(best_model)

Best Performing Model:
Model       GridSearchCV
Accuracy        0.707792
Name: 1, dtype: object


In [47]:
print("Conclusion:")
print("The Default KNN model provides a baseline performance.")
print("GridSearchCV systematically searches all specified values of K and finds the optimal hyperparameter.")
print("RandomizedSearchCV searches a random subset of K values, making it faster for larger search spaces.")
print("The model with the highest test accuracy is considered the best performer for this dataset.")

Conclusion:
The Default KNN model provides a baseline performance.
GridSearchCV systematically searches all specified values of K and finds the optimal hyperparameter.
RandomizedSearchCV searches a random subset of K values, making it faster for larger search spaces.
The model with the highest test accuracy is considered the best performer for this dataset.
